# Lab: Evaluating a Physics-Constrained Model

**Provided materials:** pre-trained PINN weights (`pinn.npz`), dataset partitions, course modules `case_physics` and `mlp`.

You are **not** training anything in this lab. The model has already been trained with a physics penalty that enforces the energy balance at every prediction. Your task is to check whether the penalty actually worked — that is, whether the model's predictions are physically consistent.

> **Guided step:** one cell below is marked `# ← COMPLETE THIS CELL`. Fill in the two indicated lines, then run all cells in order.

In [ ]:
import sys, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- module import (JupyterLite virtual filesystem) ---
# The course modules are bundled into the JupyterLite environment.
# If the import fails, the fallback below defines the key constants inline.
_modules_ok = False
for _p in ['.', '/', '/drive/v1/files']:
    try:
        if _p not in sys.path:
            sys.path.insert(0, _p)
        import case_physics, mlp
        _modules_ok = True
        print('course modules loaded from virtual filesystem')
        break
    except ImportError:
        continue

if not _modules_ok:
    print('WARNING: course modules not found on path — using inline fallback')
    # Inline fallback so the lab still runs for PoC testing
    CP_COOLANT = 4186.0
    class _Physics:
        CP_COOLANT = 4186.0
        @staticmethod
        def energy_balance_Tout(Q, mdot, Tin):
            return np.asarray(Tin) + np.asarray(Q) / (np.asarray(mdot) * CP_COOLANT)
    case_physics = _Physics()
    import types
    _mlp = types.SimpleNamespace()
    def _load(path):
        z = np.load(path)
        n = sum(1 for k in z.files if k.startswith('W'))
        class _M:
            def __init__(self):
                self.W = [z[f'W{i}'] for i in range(n)]
                self.b = [z[f'b{i}'] for i in range(n)]
            def forward(self, X):
                h = X
                for i,(W,b) in enumerate(zip(self.W,self.b)):
                    z2 = h @ W + b
                    h = np.tanh(z2) if i < len(self.W)-1 else z2
                return h
        extras = {k: z[k] for k in z.files if not (k[0] in 'Wb' and k[1:].isdigit())}
        return _M(), extras
    _mlp.load = _load
    mlp = _mlp

CP = case_physics.CP_COOLANT
INPUTS = ['Q_W', 'mdot_kgs', 'Tin_C', 'w_mm', 'k_WmK']
print(f'CP_COOLANT = {CP} J/(kg·K)')

In [ ]:
import os

# JupyterLite mounts the drive at /drive/; notebooks run from /drive/lab/
# so data files (at drive root) are one level up. Try parent dir first.
_cwd = os.getcwd()
_found = False
for _d in ['..', '/drive', '/', '/drive/v1/files', '.']:
    try:
        if os.path.exists(os.path.join(_d, 'train.csv')):
            os.chdir(_d)
            print(f'data dir: {os.path.abspath(_d)}')
            _found = True
            break
    except Exception:
        continue

if not _found:
    # Debug: show what's actually on the filesystem
    print(f'WARNING: train.csv not found. cwd was: {_cwd}')
    for _d in ['..', '/drive', '/', _cwd]:
        try:
            _ls = [f for f in os.listdir(_d) if not f.startswith('.')]
            print(f'  ls {_d!r}: {_ls[:10]}')
        except Exception as e:
            print(f'  ls {_d!r}: {e}')

# Load dataset partitions and pre-trained model
parts = {
    name: pd.read_csv(f'{name}.csv')
    for name in ('train', 'val', 'test')
}

model, scalers = mlp.load('pinn.npz')

def predict(df):
    """Run model on a dataframe, return (Tout_C, Tmax_C) array."""
    X = (df[INPUTS].values - scalers['mx']) / scalers['sx']
    return model.forward(X) * scalers['sy'] + scalers['my']

pred = {name: predict(df) for name, df in parts.items()}
print('Partition sizes:', {k: len(v) for k,v in parts.items()})
print('Model loaded. Prediction shape (test):', pred['test'].shape)

## Exercise — Energy-Balance Residual

The first-law energy balance for this system is:

$$T_{\text{out}} = T_{\text{in}} + \frac{Q}{\dot{m} \cdot c_p}$$

A physically correct model must satisfy this equation. The **energy-balance residual** is the difference between the model's predicted outlet temperature and the first-law value:

$$\text{residual} = T_{\text{out}}^{\text{predicted}} - T_{\text{out}}^{\text{first-law}}$$

A residual near zero means the model respects energy conservation. A large residual means the physics penalty did not work.

**Complete the cell below** to compute this residual for each partition.

In [ ]:
# ← COMPLETE THIS CELL
# Hint: case_physics.energy_balance_Tout(Q, mdot, Tin) returns the first-law Tout.
# pred[name][:, 0] is the model's predicted Tout for partition 'name'.

def energy_residual(df, p):
    required_Tout = case_physics.energy_balance_Tout(
        df['Q_W'].values,
        df['mdot_kgs'].values,
        df['Tin_C'].values
    )
    # ← FILL IN: subtract required_Tout from the model's predicted Tout
    return p[:, 0] - required_Tout   # replace this line

# Compute mean absolute residual for each partition
TOL = 1.0   # deg C tolerance (from the Surrogate Performance Log)
results = []
for name, df in parts.items():
    r = energy_residual(df, pred[name])
    mae = float(np.abs(r).mean())
    results.append({
        'partition': name,
        'mean |residual| (°C)': round(mae, 3),
        'max |residual| (°C)': round(float(np.abs(r).max()), 3),
        'tolerance (°C)': TOL,
        'result': 'PASS' if mae <= TOL else 'FAIL'
    })

pd.DataFrame(results)

In [ ]:
# Visualise the residual distributions
fig, ax = plt.subplots(figsize=(7, 3))
for name, df in parts.items():
    r = energy_residual(df, pred[name])
    ax.hist(r, bins=40, alpha=0.55, label=name)
ax.axvline(0, color='k', lw=0.8, label='zero (perfect)')
ax.set_xlabel('Energy-balance residual (°C)')
ax.set_ylabel('Count')
ax.set_title('A1: Energy-balance residual distribution by partition')
ax.legend()
plt.tight_layout()
plt.show()

## Interpretation

**Write your response here** (replace this text):

1. Does the model satisfy the energy-balance constraint? What does the residual distribution tell you?
2. What would a large residual on the *test* partition specifically mean for engineering use of this model?

> *Tip: focus on what the result means for the model's reliability, not just whether it passes the numeric threshold.*